In [1]:
from __future__ import annotations

import math
import re
import warnings

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d, RBFInterpolator

# ---------------------------------------------------------------------------
# Global Configuration
# ---------------------------------------------------------------------------
DATA_PATH = "/content/sample_data/dataset.csv"
XSPACE = "logm"
INTERIOR_DEG = 3
INTERIOR_NPTS = 6
ALPHA_INT = 0.7
SPIKE_ALPHA_INT = 0.6
SPIKE_THRESHOLD = 1.0
WING_POINTS = 2

LO_WING_DAMP = 0.90
HI_WING_DAMP = 1.00
SPIKE_WING_DAMP = 1.0
MIN_POINTS_PER_SIDE = 2
IV_FLOOR = 0.0
IV_CEILING = 8.0
SEP = "||"

# Expiry day config for 2D RBF handling
EXPIRY_TS = pd.Timestamp("2026-01-27 15:30:00")
EXPIRY_DATE = pd.Timestamp("2026-01-27").date()
RBF_KERNEL = "thin_plate_spline"
RBF_SMOOTH = 1e-4

# ---------------------------------------------------------------------------
# Utility Functions for Column Parsing
# ---------------------------------------------------------------------------
_DATETIME_CANDIDATES = ("datetime", "timestamp", "date_time", "date", "time")
_UNDERLYING_HINTS = ("underlying", "spot", "future", "forward", "fut")
_CE_BOUNDARY = re.compile(r"(?:^|[^a-z])(ce|call)(?:[^a-z]|$)")
_PE_BOUNDARY = re.compile(r"(?:^|[^a-z])(pe|put)(?:[^a-z]|$)")
_LEADING_C = re.compile(r"^\s*c[\s_\-]*\d")
_LEADING_P = re.compile(r"^\s*p[\s_\-]*\d")
_EXPIRY_PREFIX = re.compile(r"(?i)^[a-z]+\d{1,2}[a-z]{3}\d{2}")

def detect_datetime_col(df: pd.DataFrame) -> str:
    lower = {c.lower(): c for c in df.columns}
    for cand in _DATETIME_CANDIDATES:
        if cand in lower:
            return lower[cand]
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            return c
    return df.columns[0]

def _is_underlying(name: str) -> bool:
    low = name.lower()
    return any(h in low for h in _UNDERLYING_HINTS)

def _detect_underlying(df: pd.DataFrame, dt_col: str):
    for c in df.columns:
        if c != dt_col and _is_underlying(c):
            return c
    return None

def _extract_strike(name: str) -> float:
    stripped = _EXPIRY_PREFIX.sub("", name)
    m = re.search(r"(\d+)", stripped)
    if m:
        return float(m.group(1))
    digits = re.sub(r"[^0-9]", "", name)
    if digits:
        return float(digits[-5:]) if len(digits) > 5 else float(digits)
    return float("nan")

def _opt_type(name: str) -> str:
    low = name.lower()
    if _CE_BOUNDARY.search(low) or _LEADING_C.match(low):
        return "CE"
    if _PE_BOUNDARY.search(low) or _LEADING_P.match(low):
        return "PE"
    return "NA"

def parse_sides(df: pd.DataFrame, dt_col: str):
    infos = []
    for c in df.columns:
        if c == dt_col or _is_underlying(c):
            continue
        opt = _opt_type(c)
        strike = _extract_strike(c)
        if opt != "NA" and not math.isnan(strike):
            infos.append((c, strike, opt))

    ce = sorted((i for i in infos if i[2] == "CE"), key=lambda i: i[1])
    pe = sorted((i for i in infos if i[2] == "PE"), key=lambda i: i[1])

    return [([i[0] for i in ce], np.array([i[1] for i in ce], float)),
            ([i[0] for i in pe], np.array([i[1] for i in pe], float))]

# ---------------------------------------------------------------------------
# Cross-Sectional Imputation
# ---------------------------------------------------------------------------
def _xform(strikes: np.ndarray, F, xspace: str) -> np.ndarray:
    """Transforms strikes into log-moneyness or standard moneyness."""
    if xspace == "logm" and F and F > 0:
        return np.log(np.clip(strikes / F, 1e-9, None))
    if xspace == "money" and F and F > 0:
        return strikes / F
    return strikes.astype(float)

def reconstruct_side(strikes: np.ndarray, ivs: np.ndarray, F) -> np.ndarray:
    """Performs intra-day cross-sectional interpolation and wing extrapolation."""
    out = ivs.astype(float).copy()
    obs = np.isfinite(out)

    if obs.sum() < MIN_POINTS_PER_SIDE:
        return out

    vs, vv = strikes[obs], out[obs]
    miss = np.where(~obs)[0]

    if miss.size == 0:
        return out

    spike = float(np.nanmax(vv)) > SPIKE_THRESHOLD
    alpha = SPIKE_ALPHA_INT if spike else ALPHA_INT
    Xs = _xform(vs, F, XSPACE)
    lin = interp1d(vs, vv, kind="linear", bounds_error=False, fill_value=np.nan)

    def _local_poly(x0_native: float) -> float:
        if len(vs) < INTERIOR_DEG + 1:
            return float("nan")
        x0 = _xform(np.array([x0_native]), F, XSPACE)[0]
        idx = np.argsort(np.abs(Xs - x0))[:max(INTERIOR_NPTS, INTERIOR_DEG + 1)]
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                coef = np.polyfit(Xs[idx], vv[idx], min(INTERIOR_DEG, len(idx) - 1))
            return float(np.polyval(coef, x0))
        except Exception:
            return float("nan")

    def _wing(x0: float) -> float:
        k = int(min(WING_POINTS, len(vs)))
        low = x0 < vs[0]
        xs2, ys2 = (vs[:k], vv[:k]) if low else (vs[-k:], vv[-k:])

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            s, b = np.polyfit(xs2, ys2, 1)

        v = s * x0 + b
        dmp = SPIKE_WING_DAMP if spike else (LO_WING_DAMP if low else HI_WING_DAMP)

        if dmp != 1.0:
            anchor = vv[0] if low else vv[-1]
            v = anchor + dmp * (v - anchor)
        return float(v)

    for j in miss:
        x0 = strikes[j]
        v = float(lin(x0))

        if np.isfinite(v):
            if alpha > 0:
                q = _local_poly(x0)
                if np.isfinite(q):
                    v = (1.0 - alpha) * v + alpha * q
        else:
            v = _wing(x0)

        out[j] = min(max(v, IV_FLOOR), IV_CEILING)

    return out

def apply_imputer(df: pd.DataFrame) -> pd.DataFrame:
    """Applies cross-sectional imputation across the entire dataset."""
    out = df.copy()
    dt_col = detect_datetime_col(df)
    und_col = _detect_underlying(df, dt_col)
    sides = parse_sides(df, dt_col)
    Farr = out[und_col].to_numpy(float) if und_col else np.full(len(out), np.nan)

    all_cols = []
    for names, strikes in sides:
        if not names:
            continue
        all_cols += names
        mat = out[names].to_numpy(float).copy()

        for r in range(mat.shape[0]):
            if np.isnan(mat[r]).any():
                mat[r] = reconstruct_side(strikes, mat[r], Farr[r])
        out[names] = mat

    # Fallback causal filling for entire rows missing data
    if out[all_cols].isna().any().any():
        order = np.argsort(
            pd.to_datetime(out[dt_col], errors="coerce", dayfirst=True).to_numpy(),
            kind="stable"
        )
        tmp = out.iloc[order].copy()
        tmp[all_cols] = tmp[all_cols].ffill()
        out.iloc[order] = tmp.to_numpy()

        resid = out[all_cols].isna().any(axis=1)
        if resid.any():
            rm = out.loc[resid, all_cols].mean(axis=1)
            for c in all_cols:
                out.loc[resid, c] = out.loc[resid, c].fillna(rm)

        out[all_cols] = out[all_cols].fillna(IV_FLOOR)

    out[all_cols] = out[all_cols].clip(lower=IV_FLOOR, upper=IV_CEILING)
    return out

# ---------------------------------------------------------------------------
# Expiry Day (Jan 27) 2D RBF Override
# ---------------------------------------------------------------------------
def apply_jan27_rbf(df_orig: pd.DataFrame, df_filled: pd.DataFrame, orig_missing: pd.DataFrame) -> pd.DataFrame:
    """
    Standard cross-sectional methods degrade near expiration due to rapidly
    decaying time-to-expiry (TTE) and corresponding IV spikes.
    This function fits a joint 2D surface (log-moneyness x sqrt(TTE)) across
    all observed Jan 27 data points to preserve the terminal smile structure.
    """
    dt_col = detect_datetime_col(df_orig)
    und_col = _detect_underlying(df_orig, dt_col)
    sides = parse_sides(df_orig, dt_col)

    dt_series = pd.to_datetime(df_orig[dt_col], dayfirst=True, errors="coerce")
    tte_h = ((EXPIRY_TS - dt_series).dt.total_seconds() / 3600).clip(lower=1/60)
    sqrt_tte = np.sqrt(tte_h).to_numpy(float)
    is_expiry = (dt_series.dt.date == EXPIRY_DATE).to_numpy()

    jan27_idx = np.where(is_expiry)[0]
    if len(jan27_idx) == 0:
        return df_filled

    und_arr = df_orig[und_col].to_numpy(float) if und_col else None

    for names, strikes in sides:
        if not names:
            continue

        opt_type = _opt_type(names[0])
        obs_lm, obs_t, obs_iv = [], [], []

        # Aggregate observed points for the fitting process
        for r in jan27_idx:
            F = und_arr[r] if und_arr is not None else np.nan
            if np.isnan(F) or F <= 0:
                continue

            t = sqrt_tte[r]
            for j, col in enumerate(names):
                iv = df_orig.iat[r, df_orig.columns.get_loc(col)]
                if np.isfinite(iv):
                    obs_lm.append(np.log(strikes[j] / F))
                    obs_t.append(t)
                    obs_iv.append(iv)

        n_obs = len(obs_iv)
        if n_obs < 4:
            continue

        # Fit surface using Thin Plate Spline RBF
        rbf = RBFInterpolator(
            np.column_stack([obs_lm, obs_t]),
            np.array(obs_iv),
            kernel=RBF_KERNEL,
            smoothing=RBF_SMOOTH,
        )

        # Override predictions for originally-missing cells
        for r in jan27_idx:
            F = und_arr[r] if und_arr is not None else np.nan
            if np.isnan(F) or F <= 0:
                continue

            t = sqrt_tte[r]
            for j, col in enumerate(names):
                if orig_missing.iat[r, df_orig.columns.get_loc(col)]:
                    lm = np.log(strikes[j] / F)
                    iv_pred = float(rbf([[lm, t]]))
                    df_filled.iat[r, df_filled.columns.get_loc(col)] = \
                        float(np.clip(iv_pred, IV_FLOOR, IV_CEILING))

    return df_filled

# ===========================================================================
# Execution Pipeline
# ===========================================================================
if __name__ == "__main__":
    df = pd.read_csv(DATA_PATH)

    print("Executing cross-sectional base imputation...")
    orig_missing = df.isna()
    df_filled = apply_imputer(df)

    print("Applying Expiry Day (Jan 27) 2D RBF override...")
    df_filled = apply_jan27_rbf(df, df_filled, orig_missing)

    # Standardize datetime format for output
    dt_col = detect_datetime_col(df_filled)
    if dt_col in df_filled.columns:
        try:
            df_filled[dt_col] = df_filled[dt_col].dt.strftime('%d-%m-%Y %H:%M')
        except AttributeError:
            df_filled[dt_col] = pd.to_datetime(
                df_filled[dt_col], dayfirst=True, format='mixed'
            ).dt.strftime('%d-%m-%Y %H:%M')

    df_filled.to_csv('filled_dataset.csv', index=False)
    print("Exported standard filled dataset to filled_dataset.csv")

    # Generate flat submission file
    option_cols = []
    for names, _ in parse_sides(df, dt_col):
        option_cols.extend(names)

    rows = []
    for col in option_cols:
        missing_idx = df[col].isna()
        for idx in df.index[missing_idx]:
            dt = pd.to_datetime(
                df.loc[idx, dt_col],
                dayfirst=True
            ).strftime("%d-%m-%Y %H:%M")

            rows.append({
                "id": f"{dt}{SEP}{col}",
                "value": float(df_filled.loc[idx, col])
            })

    submission = (
        pd.DataFrame(rows)
          .sort_values("id")
          .reset_index(drop=True)
    )

    submission.to_csv("submission.csv", index=False)

    print(f"Process complete. Generated submission.csv with {submission.shape[0]} rows.")

Executing cross-sectional base imputation...
Applying Expiry Day (Jan 27) 2D RBF override...


/tmp/ipykernel_4054/3206623059.py:287: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iv_pred = float(rbf([[lm, t]]))


Exported standard filled dataset to filled_dataset.csv
Process complete. Generated submission.csv with 5460 rows.
